In [6]:
import pandas as pd

r_cols = ['user_id','movie_id', 'rating']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols, usecols=range(3))
ratings.head()

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


In [13]:
import numpy as np

# | movie_id | rating |
# | Movie 1 |    3    |
# | Movie 1 |    4    |
# | Movie 1 |    5    |
# | Movie 1 |    2    |
# | Movie 1 |    5    |

# Properties of movies will be stored by grouping the movie_id
# So now the movie_id has these ratings
# Movie 1 | 3,4,5,2,5
# where the rating column will have number of ratings(size) and Avg Rating(mean) aggregated
#In numpy there is no np.count but there is np.size

movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})
movieProperties.head()


C:\Users\Aditya\AppData\Local\Temp\ipykernel_11820\2686279558.py:16: FutureWarning: The provided callable <function mean at 0x000002810881FEC0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})


rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

In [14]:
# Create a dataframe with size inside rating multi-index
movieNumRatings = pd.DataFrame(movieProperties['rating']['size'])

# Relative to all the movies we will calculate the 'rating score' with the following formula
movieNormalizedNumRatings = movieNumRatings.apply(lambda x: (x-np.min(x)) / (np.max(x) - np.min(x)) )
movieNormalizedNumRatings.head()

,size
movie_id,
1,0.773585
2,0.222985
3,0.152659
4,0.356775
5,0.145798


In [20]:
# we will create a dictionary that maps movie_id to its 
# name, genre info, popularity(how many people rated it) and rating

movieDict = {}
with open(r'ml-100k/u.item', encoding="ISO-8859-1") as f:
    temp = ''

    # for everyline
    for line in f:

        # Remove the enter character and make a list with each element seperated by pipe | 
        fields = line.rstrip('\n').split('|')

        #convert the first element of the list into a int from string
        movieID = int(fields[0])

        # 2 element stores the name
        name = fields[1]

        # Genres are strored from 5th to 24th place
        genres = fields[5:25]
        # if it doesnt belong to that genre place 0, else place 1
        genres = map(int, genres)

        # form a key value pair, where movie_id is key 
        # and name, genre, size, rating are the values as tuple
        movieDict[movieID] = (name, np.array(list(genres)), movieNormalizedNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating.get('mean') )

print(movieDict[5])

('Copycat (1995)', array([0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]), np.float64(0.1457975986277873), np.float64(3.302325581395349))


In [27]:
# Lets make a funtion that simply calculates distance as genre and popularity

from scipy import spatial
def computeDistance(a, b):

    # outputs 0 to 2 range 
    # 0 or close to zero means high alignment in direction
    # 1 means no alignment, theyu are in 90 deg in space
    # 2 means perfectly opposite alignment
    genreDistance = spatial.distance.cosine(a[1],b[1])
    popularityDistance = abs(a[2]-b[2])
    return genreDistance + popularityDistance

computeDistance(movieDict[2], movieDict[5])

np.float64(0.7438536306460836)

### Now Actually claculate the K Nearest Neighbour using this Function

In [33]:
import operator

def getNeighbours(movieID, K):
    distances = []
    for movie in movieDict:
        # The code which prevents comparision from self
        if(movie != movieID):
            dist = computeDistance(movieDict[movieID], movieDict[movie])
            distances.append((movie, dist))

    # distances has tuples (movie_name, distance)
    # operator.itemgetter(1) considers the distance
    # Sort the distances, low to High
    distances.sort(key=operator.itemgetter(1))
    neighbours = []

    # Include K neighbours
    for x in range(K):
        # neighbours contain the movie_id
        neighbours.append(distances[x][0])
    
    return neighbours            

In [41]:
# Lets take an Example
K = 10
aggRating = 0
neighbours = getNeighbours(1, K)

#printing K Neighbours
for neighbour in neighbours:
    # movieDict has movie_id:tuples where 4th element in tuple is the avg rating of that movie
    # sum the ratings to get an aggregate rating for the K Neighboured Movies
    aggRating += movieDict[neighbour][3]
    print(f"{movieDict[neighbour][0]} has rating: {movieDict[neighbour][0]}")

avgRating = aggRating/K


Liar Liar (1997) has rating: Liar Liar (1997)
Aladdin (1992) has rating: Aladdin (1992)
Willy Wonka and the Chocolate Factory (1971) has rating: Willy Wonka and the Chocolate Factory (1971)
Monty Python and the Holy Grail (1974) has rating: Monty Python and the Holy Grail (1974)
Full Monty, The (1997) has rating: Full Monty, The (1997)
George of the Jungle (1997) has rating: George of the Jungle (1997)
Beavis and Butt-head Do America (1996) has rating: Beavis and Butt-head Do America (1996)
Birdcage, The (1996) has rating: Birdcage, The (1996)
Home Alone (1990) has rating: Home Alone (1990)
Aladdin and the King of Thieves (1996) has rating: Aladdin and the King of Thieves (1996)


In [42]:
avgRating

np.float64(3.3445905900235564)